### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="fitness_club",
    dataset_year="2023",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/ddosad/datacamps-data-science-associate-certification",
    download_description="""
kaggle datasets download -d ddosad/datacamps-data-science-associate-certification -p local-data-warehouse/fitness_club/ --unzip
""",
    # References
    academic_reference_bibtex=r"""@misc{ddosad2023fitness,
  author       = {Kaggle User Ddosad},
  title        = {Fitness Club Dataset for ML Classification},
  year         = {2023},
  howpublished = {\url{https://www.kaggle.com/datasets/ddosad/datacamps-data-science-associate-certification}},
  note         = {Kaggle dataset},
}
""",
    academic_reference_bibtex_key="ddosad2023fitness",
    license="Public Domain",
    data_tags=["IID"],
    curation_comments="""
- We dropped the booking_id column.
- We renamed the values of the target variable to be more meaningful ("Yes"/"No").
- We treat "-0" values as "0" values for the target, following the description.
- We removed trailing words (like "days") from "days_before".
- We aligned the naming of "day_of_week" to be consistent per day.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="attended",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="attended",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/fitness_class_2212.csv")

df = df.drop(columns=["booking_id"])

target_feature = "attended"
df[target_feature] = abs(df[target_feature])
df[target_feature] = df[target_feature].map({0: "No", 1: "Yes"})

# Remove trailing text from column
df["days_before"] = df["days_before"].str.replace(" days", "").astype(int)
df["day_of_week"] = (
    df["day_of_week"]
    .str.replace("Wednesday", "Wed")
    .replace("Monday", "Mon")
    .replace("Fri.", "Fri")
)

cat_features = [
    "day_of_week",
    "time",
    "category",
    "attended",
]

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df[cat_features] = df[cat_features].astype("category")

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 1,500
Columns: 7
Use sampling: False (sample size: 1,500)
Get row duplicates (staged, merged)...
Using top-6 columns for initial filtering: ['weight', 'months_as_member', 'days_before', 'day_of_week', 'category', 'time']
Rows remaining as candidates after top-6 filter: 2 (of 1,500)

#### Duplicate Report
Total duplicate rows: 1 (0.07% of dataset)
Duplicate rows ignoring target: 1 (0.07% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,months_as_member,weight,days_before,day_of_week,time,category,attended
0,15,65.47,6,Wed,AM,HIIT,Yes
1,18,77.85,8,Thu,AM,Strength,Yes
2,13,67.26,10,Fri,AM,Cycling,No
3,7,86.70,12,Sat,AM,HIIT,No
4,5,135.18,8,Thu,AM,HIIT,No


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,day_of_week,category,0.0,0.00,7.0,"Fri, Thu, Mon, Sun, Sat, Tue, Wed"
1,time,category,0.0,0.00,2.0,"AM, PM"
2,category,category,0.0,0.00,6.0,"HIIT, Cycling, Strength, Yoga, Aqua, -"
3,attended,category,0.0,0.00,2.0,"No, Yes"
4,weight,float64,20.0,1.33,1241.0,"78.28, 84.64, 75.63, 79.68, 83.35, 76.18, 69.0, 82.63, 89.99, 71.74"
5,months_as_member,int64,0.0,0.00,72.0,"8, 7, 6, 9, 12, 5, 11, 10, 13, 15"
6,days_before,int64,0.0,0.00,19.0,"10, 2, 8, 12, 14, 4, 6, 7, 3, 5"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
months_as_member,1500.0,15.628667,12.926543,1.00,148.00
weight,1480.0,82.610378,12.765859,55.41,170.52
days_before,1500.0,8.346667,4.077938,1.00,29.00


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column      rank                        
attended    1           No   1046  69.73
            2          Yes    454  30.27
category    1         HIIT    667  44.47
            2      Cycling    376  25.07
            3     Strength    233  15.53
            4         Yoga    135   9.00
            5         Aqua     76   5.07
day_of_week 1          Fri    305  20.33
            2          Thu    241  16.07
            3          Mon    228  15.20
            4          Sun    213  14.20
            5          Sat    202  13.47
time        1           AM   1141  76.07
            2           PM    359  23.93

In [8]:
# Target Distribution
target_df

,count,pct
attended,,
No,1046,69.73
Yes,454,30.27


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=10, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...


Saving curated container to fitness_club/019d7367-aced-7960-94c9-b19b393cb0ad
019d7367-aced-7960-94c9-b19b393cb0ad
34ae21db3ea6103fe9fa3def0ca2db184970671526bbbf349da6df4d5c42ce2b
